# Exploring CMS PFNanoAOD with `uproot` and `awkward`

**A hands-on tutorial: from opening a ROOT file to linking particle-flow candidates with jets**

In this notebook you will:

1. Open a CMS NanoAOD file with **`uproot`**, without installing ROOT.
2. Look at what's inside: trees, branches, and data types.
3. Load data as **`awkward`** arrays and learn to work with *jagged* data (a different number of jets, muons, … in every event).
4. Make basic histograms with `matplotlib`.
5. Compute kinematic quantities ($\Delta\phi$, $\Delta R$, invariant mass) by hand.
6. **Associate PF candidates (`PFCands`) with the jets they belong to**, and check the association against what CMS stored for each jet.

Run the cells in order with `Shift+Enter`. Read the text, run the code, and then *change something* and run it again; that's how this sticks. Exercises are marked **✏️ Try it**.

The input file, `nano_data2016_2-2.root`, should sit in the same folder as this notebook.

---
**Contents**

0. Setup
1. Background: NanoAOD, PFNano, PF candidates, and jets
2. Opening the file with `uproot`
3. Reading branches into `awkward` arrays
4. Building "collections": jets and PF candidates as objects
5. Selections and masks
6. First plots
7. Kinematics by hand: $\Delta\phi$, $\Delta R$, invariant mass
8. An event display in the $(\eta, \phi)$ plane
9. Associating PF candidates with jets
10. Checking the association, and studying jet constituents
11. Running over the whole file in chunks
12. More exercises
13. References and further reading

## 0. Setup

You need four Python packages. If any import below fails, install it from a terminal (or uncomment the `%pip` line and run it once):

```
pip install uproot awkward numpy matplotlib
```

You **don't** need ROOT, CMSSW, or `coffea`. `uproot` is pure Python and reads ROOT files directly.

In [ ]:
# %pip install uproot awkward numpy matplotlib

import time
import numpy as np
import awkward as ak
import uproot
import matplotlib.pyplot as plt

print("uproot ", uproot.__version__)
print("awkward", ak.__version__)
print("numpy  ", np.__version__)

FILENAME = "nano_data2016_2-2.root"   # the PFNanoAOD file you were given

# Plot defaults: clean axes, readable fonts
plt.rcParams.update({
    "figure.figsize": (7, 4.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

# One fixed set of colors for particle types, used in every plot
TYPE_COLORS = {
    "charged hadron": "#2a78d6",
    "neutral hadron": "#eb6834",
    "photon":         "#1baf7a",
    "electron":       "#eda100",
    "muon":           "#e87ba4",
    "HF (forward)":   "#8a8a85",
}

## 1. Background

### NanoAOD
CMS stores data in a series of formats that get smaller and simpler at each step: RAW → AOD → MiniAOD → **NanoAOD**. NanoAOD is a *flat* format. Every quantity is a simple number or a list of numbers per event, stored in one ROOT `TTree` called `Events`. Because it's flat, you can read it with `uproot` and analyze it with array tools, with no CMS software needed.

The CERN Open Data portal has a good introduction: [Getting started with CMS NanoAOD](https://opendata.cern.ch/docs/cms-getting-started-nanoaod).

### How "jagged" data is stored
Each event has a different number of jets. NanoAOD stores this as a **counter** branch plus one branch per property:

| branch | meaning | shape |
|---|---|---|
| `nJet` | number of jets in this event | one integer per event |
| `Jet_pt`, `Jet_eta`, `Jet_phi`, … | jet properties | a list of length `nJet` per event |

The same pattern holds for every collection: `nMuon` + `Muon_*`, `nPFCands` + `PFCands_*`, and so on. `awkward` arrays are built for exactly this "list of lists with different lengths" structure.

### PFNano: NanoAOD plus particle-flow candidates
Standard NanoAOD keeps *reconstructed objects* (jets, muons, electrons, …) but throws away the individual particles that jets are made of. **PFNano** is an extension that also writes out the **particle-flow (PF) candidates**.

CMS's *particle-flow* algorithm combines information from every subdetector (tracker, calorimeters, muon system) to identify each individual particle in the event as one of five types: charged hadron, neutral hadron, photon, electron, or muon. Those are the PF candidates. Jets are built by clustering PF candidates.

Your file was made with the CMS Open Data [PFNano producer tool for 2016](https://opendata.cern.ch/record/12504) ([source code on GitHub](https://github.com/cms-opendata-analyses/PFNanoProducerTool)). Its default configuration stores **all** PF candidates in the event in a collection called `PFCands`.

### The PF candidate type: `PFCands_pdgId`
The candidate's "type" is stored as a [PDG particle code](https://pdg.lbl.gov/2024/reviews/rpp2024-rev-monte-carlo-numbering.pdf). Keep in mind that this is the reconstruction's best guess, *not* the true particle identity:

| `abs(pdgId)` | PF type |
|---|---|
| 211 | charged hadron (mostly π±, but also K±, p, …) |
| 130 | neutral hadron (mostly K⁰_L, neutrons, …) |
| 22 | photon (mostly from π⁰ → γγ) |
| 11 | electron |
| 13 | muon |
| 1, 2 | hadronic / EM deposit in the forward (HF) calorimeter, \|η\| > 3 |

### Jets in this file
`Jet` holds **AK4 CHS** jets: the anti-$k_T$ algorithm with radius parameter $R = 0.4$, run on PF candidates after **C**harged **H**adron **S**ubtraction. CHS removes charged candidates that are associated with *pileup* vertices, meaning other proton–proton collisions in the same bunch crossing. Jet momenta are calibrated (jet energy corrections), and only jets with $p_T > 15$ GeV are kept. `FatJet` holds the larger AK8 ($R = 0.8$) jets.

### Coordinates
CMS uses $p_T$ (momentum transverse to the beam), pseudorapidity $\eta = -\ln\tan(\theta/2)$, and azimuthal angle $\phi$. Angular distance between two objects is

$$\Delta R = \sqrt{(\Delta\eta)^2 + (\Delta\phi)^2}.$$

Roughly speaking, an AK4 jet is a cone of radius $\Delta R \approx 0.4$ around its axis.

## 2. Opening the file with `uproot`

`uproot.open` gives you a file object that works like a Python dictionary. Opening it does **not** read the data yet, so this is fast even for a large file.

In [ ]:
f = uproot.open(FILENAME)
print(f.keys())        # top-level objects in the file

You should see `Events` (the data) and a few bookkeeping trees (`Runs`, `LuminosityBlocks`, and so on). The `;1` is a ROOT "cycle" number that you can ignore.

In [ ]:
tree = f["Events"]
print("Number of events:  ", tree.num_entries)
print("Number of branches:", len(tree.keys()))

NanoAOD has **over a thousand branches**, so you need to search for the ones you want. `filter_name` accepts wildcards:

In [ ]:
print(tree.keys(filter_name="nJet"))
print(tree.keys(filter_name="Jet_*"))

In [ ]:
print(tree.keys(filter_name="*PFCands*"))

`tree.show()` prints each branch's **type** and **documentation string**. That's your quick-reference "dictionary" for what a variable means:

In [ ]:
tree.show(filter_name=["nPFCands", "PFCands_*"])

In [ ]:
tree.show(filter_name=["nJet", "Jet_pt", "Jet_eta", "Jet_phi", "Jet_mass",
                       "Jet_nConstituents", "Jet_rawFactor", "Jet_jetId",
                       "Jet_chHEF", "Jet_neHEF", "Jet_neEmEF", "Jet_chEmEF", "Jet_muEF"])

> **Reference for every branch.** The full list of branches in a 2016 PFNano data file, with types and descriptions, is in the producer tool's documentation page: [`nanoAOD_2016data.html`](https://htmlpreview.github.io/?https://github.com/cms-opendata-analyses/PFNanoProducerTool/blob/master/nanoAOD_2016data.html) (the [raw file](https://github.com/cms-opendata-analyses/PFNanoProducerTool/blob/master/nanoAOD_2016data.html) is in the repo). The [CMS NanoAOD documentation](https://cms-nanoaod-integration.web.cern.ch/autoDoc/) covers standard NanoAOD branches for many releases. In the end, `tree.show()` on *your* file is the final word.

**✏️ Try it:** List all the muon branches. Then find the trigger branches for single-jet triggers (hint: `filter_name="HLT_PFJet*"`).

## 3. Reading branches into `awkward` arrays

`tree.arrays(...)` actually reads data. Two rules keep it fast:

* **Only read the branches you need** (`filter_name=` or an explicit list).
* **Only read as many events as you need** while you're developing (`entry_stop=`).

The PF candidate branches are by far the largest part of the file (thousands of candidates per event), so we start with 5,000 events.

In [ ]:
N_EVENTS = 5000

t0 = time.time()
events = tree.arrays(
    filter_name=["run", "luminosityBlock", "event", "PV_npvs",
                 "nJet", "Jet_*", "nPFCands", "PFCands_*"],
    entry_stop=N_EVENTS,
)
print(f"Read {len(events)} events in {time.time() - t0:.1f} s")

`events` is an **awkward Array**. Each element is one event, and each event is a *record* with named fields:

In [ ]:
print(events.fields[:15], "...")
events.type.show()

Read the type like a sentence. `5000 * {run: uint32, ..., Jet_pt: var * float32, ...}` means: *5000 events, each a record. `run` is one number, while `Jet_pt` is a **var**iable-length list of floats.*

Field access works with a dot or with square brackets:

In [ ]:
print(events.nJet)               # one number per event
print(events.Jet_pt)             # a list per event
print(events["Jet_pt"][0])       # the jet pTs in the first event

### Indexing jagged arrays
The first index picks **events** and the second picks **objects within each event**:

In [ ]:
print("first event, all jets:   ", events.Jet_pt[0])
print("first event, first jet:  ", events.Jet_pt[0, 0])
print("first 5 events, all jets:", events.Jet_pt[:5].tolist())

### Counting and flattening
* `ak.num(x)` gives the length of each event's list.
* `ak.flatten(x)` removes the event structure and returns one long array of *all* objects. You need this for histograms of object-level quantities.

In [ ]:
print("ak.num:     ", ak.num(events.Jet_pt)[:10])
print("nJet branch:", events.nJet[:10])          # same thing, stored in the file

all_jet_pt = ak.flatten(events.Jet_pt)
print(f"{len(all_jet_pt)} jets in {len(events)} events")

### Vectorized math
NumPy functions work element by element on awkward arrays, **keeping the jagged structure**. You almost never need a `for` loop over events, and you should avoid them because they're very slow in Python.

In [ ]:
jet_px = events.Jet_pt * np.cos(events.Jet_phi)
print(jet_px[:3].tolist())

# Reductions along the inner axis: one number per event
print("scalar sum of jet pT (HT):", ak.sum(events.Jet_pt, axis=1)[:5])
print("max jet pT per event:     ", ak.max(events.Jet_pt, axis=1)[:5])

**✏️ Try it:** What's the largest number of PF candidates in any of these 5,000 events? The smallest? (Hint: `ak.num`, `ak.max`, `ak.min`, or just `events.nPFCands`.)

**✏️ Try it:** Pick one event and use a plain Python `for` loop to print each jet's $p_T$, $\eta$, and $\phi$. Then do the same with `ak.zip` (next section) and `.tolist()`.

## 4. Building collections

Writing `events.Jet_pt`, `events.Jet_eta`, … gets tedious. It's much nicer to have a `jets` array where each jet is a record with `.pt`, `.eta`, `.phi`, and so on. `ak.zip` does this: it combines several arrays **with the same jagged structure** into one array of records.

This is (a simple version of) what `coffea`'s `NanoEvents` does for you automatically.

In [ ]:
def get_collection(events, name):
    # Zip all branches called '<name>_<something>' into one awkward array of records.
    prefix = name + "_"
    fields = {k[len(prefix):]: events[k] for k in events.fields if k.startswith(prefix)}
    return ak.zip(fields)

jets    = get_collection(events, "Jet")
pfcands = get_collection(events, "PFCands")

print(jets.fields)
print(pfcands.fields)

In [ ]:
print(jets[0].tolist()[:2])     # the first two jets of event 0, as Python dictionaries
print(jets.pt[0], jets[0].pt)   # two ways of writing the same thing

⚠️ **A common pitfall:** `jets.pt[:, 0]` ("the leading jet in every event") **crashes** if any event has zero jets. Two safe alternatives:

* `ak.firsts(jets.pt)` returns `None` for events with no jets.
* Select events with at least one jet *first*, then index.

NanoAOD jets are already sorted by $p_T$ (highest first), so index 0 is the leading jet.

In [ ]:
lead_pt = ak.firsts(jets.pt)
print(lead_pt[:10])
print("events with no jets:", ak.sum(ak.is_none(lead_pt)))

## 5. Selections and masks

A **mask** is an array of `True`/`False` values. You can use one in two different ways, and you need to keep them straight:

* An **event-level mask** has one boolean per event. Indexing with it *removes whole events*.
* An **object-level mask** has one boolean per object, with the same jagged shape as `jets`. Indexing with it *removes objects inside each event*, but every event stays (possibly with an empty list).

### A standard jet selection
* $p_T > 30$ GeV
* $|\eta| < 2.4$ (inside the tracker, so the jets have charged-particle information)
* passes the **tight jet ID**. `Jet_jetId` is a *bit field*: bit 1 (value 2) means tight. We test it with the bitwise AND operator `&`.

In [ ]:
good_jet_mask = (jets.pt > 30) & (abs(jets.eta) < 2.4) & ((jets.jetId & 2) > 0)
print(good_jet_mask[:3].tolist())          # object-level: same shape as jets

good_jets = jets[good_jet_mask]
print("jets per event before:", ak.num(jets)[:8].tolist())
print("jets per event after: ", ak.num(good_jets)[:8].tolist())

Note the parentheses: `&` binds more tightly than `>` in Python, so `(a > 1) & (b < 2)` needs all of them.

Now an **event-level** selection: keep events with at least two good jets.

In [ ]:
dijet_mask = ak.num(good_jets) >= 2        # one boolean per event
print(f"{ak.sum(dijet_mask)} of {len(events)} events have >= 2 good jets")

dijets = good_jets[dijet_mask]             # now dijets[:, 0] and dijets[:, 1] are safe
print(dijets.pt[:, 0][:5], dijets.pt[:, 1][:5])

### Selecting PF candidates by type

In [ ]:
abs_id = abs(pfcands.pdgId)

charged_hadrons = pfcands[abs_id == 211]
photons         = pfcands[abs_id == 22]
neutral_hadrons = pfcands[abs_id == 130]

print("mean # per event: charged hadrons %.0f, photons %.0f, neutral hadrons %.0f" % (
    ak.mean(ak.num(charged_hadrons)), ak.mean(ak.num(photons)), ak.mean(ak.num(neutral_hadrons))))

# The charge should match the type: charged hadrons have charge ±1, photons 0
print("charges of charged hadrons:", np.unique(ak.to_numpy(ak.flatten(charged_hadrons.charge))))
print("charges of photons:        ", np.unique(ak.to_numpy(ak.flatten(photons.charge))))

**✏️ Try it:** How many PF muons with $p_T > 20$ GeV are there in total? In how many *events* is there at least one?

## 6. First plots

To histogram an object-level quantity: **flatten** it, turn it into a NumPy array, and pass it to `plt.hist` (or `np.histogram`). A small helper saves some typing.

> `matplotlib` is all we need here. Many HEP physicists also use [`hist`](https://hist.readthedocs.io/) for histogram objects and [`mplhep`](https://mplhep.readthedocs.io/) for CMS-style plots. Both are worth learning later.

In [ ]:
def to_np(x):
    # Flatten any awkward array completely and return a NumPy array.
    return ak.to_numpy(ak.flatten(x, axis=None))

def hist1d(values, bins, xlabel, ylabel="Count", logy=False, label=None, ax=None, **kw):
    ax = ax or plt.gca()
    ax.hist(to_np(values), bins=bins, histtype="step", linewidth=2, label=label, **kw)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if logy:
        ax.set_yscale("log")
    return ax

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
hist1d(jets.pt,  np.linspace(0, 1000, 101), r"Jet $p_T$ [GeV]", logy=True, ax=axes[0])
hist1d(jets.eta, np.linspace(-5, 5, 51),    r"Jet $\eta$",  ax=axes[1])
hist1d(jets.phi, np.linspace(-np.pi, np.pi, 41), r"Jet $\phi$", ax=axes[2])
axes[0].set_title("All jets (pT > 15 GeV)")
plt.tight_layout(); plt.show()

Things to notice:
* The $p_T$ spectrum falls steeply, which is why we use a log scale. If this is a jet-triggered dataset, you may see a "turn-on" shape where the trigger threshold sits.
* The $\eta$ distribution extends to $|\eta| \approx 4.7$, the edge of the forward calorimeter. Structure near $|\eta| \approx 2.5$–$3$ comes from detector transitions.
* $\phi$ should be roughly flat, since the detector is (mostly) symmetric around the beam.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hist1d(events.nJet,     np.arange(-0.5, 25.5, 1), "Jets per event", ax=axes[0])
hist1d(events.nPFCands, 60, "PF candidates per event", ax=axes[1])
plt.tight_layout(); plt.show()

The number of PF candidates per event is in the thousands. Most of them are soft (low $p_T$) particles from the underlying event and from pileup. Let's look at their $p_T$ and type:

In [ ]:
def pf_type(pdgId):
    # Map pdgId to a PF type name (works on a NumPy array).
    a = np.abs(pdgId)
    names = np.full(a.shape, "other", dtype=object)
    names[a == 211] = "charged hadron"
    names[a == 130] = "neutral hadron"
    names[a == 22]  = "photon"
    names[a == 11]  = "electron"
    names[a == 13]  = "muon"
    names[(a == 1) | (a == 2)] = "HF (forward)"
    return names

types_flat = pf_type(to_np(pfcands.pdgId))
pt_flat    = to_np(pfcands.pt)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left: how many candidates of each type
labels = [t for t in TYPE_COLORS if (types_flat == t).any()]
counts = [np.sum(types_flat == t) / len(events) for t in labels]
axes[0].barh(labels, counts, color=[TYPE_COLORS[t] for t in labels])
axes[0].invert_yaxis()
axes[0].set_xlabel("Average number per event")
for y, c in enumerate(counts):
    axes[0].text(c, y, f" {c:.1f}" if c < 10 else f" {c:.0f}", va="center")

# Right: pT spectrum per type
bins = np.logspace(-1, 3, 61)
for t in labels:
    axes[1].hist(pt_flat[types_flat == t], bins=bins, histtype="step", linewidth=2,
                 color=TYPE_COLORS[t], label=t)
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel(r"PF candidate $p_T$ [GeV]"); axes[1].set_ylabel("Count")
axes[1].legend(frameon=False, fontsize=10)
plt.tight_layout(); plt.show()

**✏️ Try it:** Plot the $\eta$ distribution of PF candidates, separately for charged and neutral ones. Why do charged candidates stop at $|\eta| \approx 2.5$? (Think about where the tracker ends.)

**✏️ Try it:** Plot `PFCands_dz` for charged hadrons only. This is the candidate's track distance from the primary vertex along the beam line, in cm. The long tails come from pileup. Use a log y-axis and a range of about $\pm 20$ cm.

## 7. Kinematics by hand

It's worth writing these functions yourself once so you know exactly what they do. They work on awkward arrays of any shape. (The [`vector`](https://vector.readthedocs.io/) package does all of this for you, and you'll probably want it later.)

**$\Delta\phi$ must wrap around.** $\phi = 3.1$ and $\phi = -3.1$ are only 0.08 rad apart, not 6.2. The expression below maps any difference into $[-\pi, \pi)$.

In [ ]:
def delta_phi(phi1, phi2):
    return (phi1 - phi2 + np.pi) % (2 * np.pi) - np.pi

def delta_r(eta1, phi1, eta2, phi2):
    return np.sqrt((eta1 - eta2) ** 2 + delta_phi(phi1, phi2) ** 2)

def to_cartesian(pt, eta, phi, mass):
    # (pt, eta, phi, m) -> (px, py, pz, E)
    px = pt * np.cos(phi)
    py = pt * np.sin(phi)
    pz = pt * np.sinh(eta)
    E  = np.sqrt(px**2 + py**2 + pz**2 + mass**2)
    return px, py, pz, E

def invariant_mass(px, py, pz, E):
    m2 = E**2 - px**2 - py**2 - pz**2
    return np.sqrt(np.maximum(m2, 0))     # guard against tiny negative values from rounding

print(delta_phi(3.1, -3.1))               # should be about -0.083, not 6.2

### Example: the dijet invariant mass
Add the four-momenta of the two leading good jets and compute the invariant mass of the pair. We use `dijets` from Section 5 (events with at least two good jets).

In [ ]:
j1, j2 = dijets[:, 0], dijets[:, 1]
p1 = to_cartesian(j1.pt, j1.eta, j1.phi, j1.mass)
p2 = to_cartesian(j2.pt, j2.eta, j2.phi, j2.mass)
mjj = invariant_mass(*[a + b for a, b in zip(p1, p2)])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
hist1d(mjj, np.linspace(0, 3000, 61), r"$m_{jj}$ [GeV]", logy=True, ax=axes[0])
hist1d(abs(delta_phi(j1.phi, j2.phi)), np.linspace(0, np.pi, 41),
       r"$|\Delta\phi(j_1, j_2)|$", ax=axes[1])
plt.tight_layout(); plt.show()

The two leading jets tend to be **back to back** in $\phi$ ($\Delta\phi \approx \pi$) because of momentum conservation in the transverse plane. Events far from $\pi$ usually have a third hard jet.

## 8. An event display in the $(\eta, \phi)$ plane

Before writing any association code, *look* at an event. Below, each PF candidate is a dot (area ∝ $p_T$, color = type), and each good jet is drawn as a circle of radius 0.4 around its axis.

In [ ]:
def event_display(i, jets_ev, pf_ev, pt_min=0.5, R=0.4, ax=None):
    # Draw PF candidates and jet circles for one event.
    ax = ax or plt.gca()
    pf_ev = pf_ev[pf_ev.pt > pt_min]
    types = pf_type(ak.to_numpy(pf_ev.pdgId))
    for t, c in TYPE_COLORS.items():
        sel = types == t
        if sel.any():
            ax.scatter(ak.to_numpy(pf_ev.eta)[sel], ak.to_numpy(pf_ev.phi)[sel],
                       s=4 + 3 * ak.to_numpy(pf_ev.pt)[sel], color=c, alpha=0.7,
                       edgecolor="white", linewidth=0.5, label=t)
    for k, j in enumerate(jets_ev):
        circ = plt.Circle((j.eta, j.phi), R, fill=False, linewidth=2, color="black")
        ax.add_patch(circ)
        ax.text(j.eta, j.phi + R + 0.08, f"jet {k}: {j.pt:.0f} GeV",
                ha="center", fontsize=9)
    ax.set_xlim(-5, 5); ax.set_ylim(-np.pi, np.pi)
    ax.set_xlabel(r"$\eta$"); ax.set_ylabel(r"$\phi$")
    ax.set_title(f"Event index {i} (PF candidates with pT > {pt_min} GeV)")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9,
              markerscale=1.5)
    ax.set_aspect("equal")
    return ax

# pick the first event with at least 2 good jets
i_show = int(np.argmax(ak.to_numpy(dijet_mask)))
plt.figure(figsize=(12, 5))
event_display(i_show, good_jets[i_show], pfcands[i_show])
plt.show()

You should see dense clusters of candidates inside the jet circles and a sprinkling of soft candidates everywhere else (pileup and underlying event).

Two subtleties show up in the picture:
* Jets near $\phi = \pm\pi$ wrap around, and the circle gets cut at the edge of the plot. The *physics* is fine as long as $\Delta\phi$ wraps correctly.
* Anti-$k_T$ jets are **close to** circular cones of radius 0.4, but not exactly, especially when two jets are close together.

**✏️ Try it:** Change `i_show` and look at several events. Raise `pt_min` to 2 GeV. What's left?

## 9. Associating PF candidates with jets

Goal: for every jet, get **the list of PF candidates that make it up**, as an awkward array with structure

```
constituents[event][jet][candidate]
```

There are two ways to do this, depending on what's in the file.

### 9a. Option 1: an index ("association") table, if the file has one
Some PFNano configurations also write a table that records, for every (jet, candidate) pair, which jet and which candidate it is:

| branch | meaning |
|---|---|
| `nJetPFCands` | number of (jet, candidate) pairs in the event |
| `JetPFCands_jetIdx` | index into the `Jet` collection |
| `JetPFCands_pFCandsIdx` | index into the `PFCands` collection (older versions call it `candIdx`) |

Similar tables exist for AK8 jets (`FatJetPFCands_*`) and for jet secondary vertices (`JetSVs_*`). This is *exact*: it's the jet's real constituent list from CMSSW. `coffea` wraps it in an [`AssociatedPFCand`](https://scikit-hep.org/coffea/api/coffea.nanoevents.methods.nanoaod.AssociatedPFCand.html) object (see the [`PFNanoAODSchema`](https://coffea-hep.readthedocs.io/en/v0.7.23/api/coffea.nanoevents.PFNanoAODSchema.html) docs for the list of cross-references). We'll do the same thing by hand.

### 9b. Option 2: geometric ($\Delta R$) matching, if it doesn't
The **default 2016 Open Data PFNano configuration** (`onlyaddPFcands`) stores *all* PF candidates but **no** association table (check `JetPFCands` in the [branch documentation](https://htmlpreview.github.io/?https://github.com/cms-opendata-analyses/PFNanoProducerTool/blob/master/nanoAOD_2016data.html): it isn't there). In that case we rebuild the association geometrically:

> Assign each PF candidate to the **nearest** jet in $\Delta R$, provided that $\Delta R < 0.4$.

"Nearest" guarantees that each candidate belongs to at most one jet, which is also true of anti-$k_T$ clustering. This is an *approximation*. In Section 10 we measure how good it is. Later, when you recluster the candidates yourself with FastJet, you'll get the exact answer for *your* jets.

Let's see which case we're in:

In [ ]:
idx_branch = None
for name in ["JetPFCands_pFCandsIdx", "JetPFCands_candIdx"]:
    if name in tree.keys():
        idx_branch = name

HAS_ASSOC_TABLE = idx_branch is not None
if HAS_ASSOC_TABLE:
    print("This file HAS a jet-constituent index table:", idx_branch)
else:
    print("No JetPFCands table in this file -> we'll use Delta-R matching.")

### The key `awkward` tool: `ak.cartesian`
To compare *every* candidate with *every* jet in the same event, we form all (candidate, jet) pairs. With `nested=True`, the result is grouped by candidate:

```
pairs[event][candidate][jet]
```

so that taking the minimum over the last axis (`axis=2`) gives "the nearest jet for each candidate."

Here's a tiny example first, so you can see the shapes:

In [ ]:
a = ak.Array([[1, 2, 3], [4]])
b = ak.Array([["x", "y"], ["z"]])
print(ak.cartesian([a, b]).tolist())               # flat list of pairs per event
print(ak.cartesian([a, b], nested=True).tolist())  # grouped by elements of a

### Memory warning
In an event with 1,500 candidates and 10 jets, that's 15,000 pairs, and over a few thousand events it adds up quickly. So we:
* work on a smaller slice of events here (the full file is handled in chunks in Section 11), and
* match against **all** jets in the event (not only the good ones), so that a candidate that belongs to a soft jet isn't wrongly given to a harder neighbor.

In [ ]:
N_MATCH = 1000                      # number of events to use for the association
jets_m = jets[:N_MATCH]
pf_m   = pfcands[:N_MATCH]

def match_pf_to_jets(pf, jets, R=0.4):
    # For each PF candidate, return the index of the nearest jet with dR < R, or -1.
    # Output has the same jagged shape as pf: [event][candidate].
    pairs = ak.cartesian({"pf": pf, "jet": jets}, nested=True)   # [event][cand][jet]
    dr = delta_r(pairs.pf.eta, pairs.pf.phi, pairs.jet.eta, pairs.jet.phi)
    nearest = ak.argmin(dr, axis=2)            # index of the closest jet (None if no jets)
    dr_min  = ak.min(dr, axis=2)
    inside  = ak.fill_none(dr_min < R, False)
    return ak.where(inside, ak.fill_none(nearest, -1), -1), dr_min

t0 = time.time()
pf_jet_idx, pf_dr_min = match_pf_to_jets(pf_m, jets_m)
print(f"matched in {time.time() - t0:.1f} s")

pf_m = ak.with_field(pf_m, pf_jet_idx, "jetIdx")   # attach the result as a new field
print(pf_m.jetIdx[0][:30].tolist())
print("fraction of candidates inside some jet: %.3f" % ak.mean(pf_m.jetIdx >= 0))

Now `pf_m.jetIdx` tells us which jet (by its position in the `Jet` collection) each candidate belongs to, with `-1` meaning none. This has exactly the same information as the `JetPFCands_jetIdx` / `JetPFCands_pFCandsIdx` table from Option 1, organized from the candidates' point of view.

### Grouping candidates by jet
Next we turn "a jet index for each candidate" into "a list of candidates for each jet." Once more we pair every jet *index* with every candidate, then keep only the pairs where they agree:

In [ ]:
def group_by_jet(jets, cands):
    # cands must have a 'jetIdx' field. Returns cands arranged as [event][jet][candidate].
    jet_index = ak.local_index(jets, axis=1)                         # 0, 1, 2, ... per event
    grid = ak.cartesian({"j": jet_index, "c": cands}, nested=True)  # [event][jet][cand]
    return grid.c[grid.j == grid.c.jetIdx]

constituents = group_by_jet(jets_m, pf_m)
constituents.type.show()

In [ ]:
ev, jt = i_show, 0
print(f"event {ev}, jet {jt}: pT = {jets_m[ev, jt].pt:.1f} GeV, "
      f"{len(constituents[ev, jt])} matched candidates")
print("their pT:   ", np.round(ak.to_numpy(constituents[ev, jt].pt), 1)[:20], "...")
print("their pdgId:", ak.to_numpy(constituents[ev, jt].pdgId)[:20], "...")

### 9c. Using the index table instead (for files that have one)
If `HAS_ASSOC_TABLE` is `True`, the same `group_by_jet` function works. We just fill `jetIdx` from the table instead of from $\Delta R$. Awkward lets you index a jagged array with a jagged array of integers (`pf[idx]`), one event at a time and all at once.

In [ ]:
if HAS_ASSOC_TABLE:
    assoc = tree.arrays(["JetPFCands_jetIdx", idx_branch], entry_stop=N_MATCH)
    cands_from_table = pfcands[:N_MATCH][assoc[idx_branch]]          # look up each candidate
    cands_from_table = ak.with_field(cands_from_table, assoc.JetPFCands_jetIdx, "jetIdx")
    constituents_exact = group_by_jet(jets_m, cands_from_table)
    print("exact constituent counts, event", i_show, ":",
          ak.num(constituents_exact[i_show]).tolist())
    print("Delta-R constituent counts, event", i_show, ":",
          ak.num(constituents[i_show]).tolist())
else:
    print("No association table in this file; skipping (the Delta-R result is used below).")

### Drawing the result
Here's the event display again, now coloring only the candidates assigned to each jet:

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
loose = pf_m[i_show][(pf_m[i_show].jetIdx < 0) & (pf_m[i_show].pt > 0.5)]
ax.scatter(ak.to_numpy(loose.eta), ak.to_numpy(loose.phi), s=4, color="#c8c8c3",
           label="not in any jet")
jet_colors = list(TYPE_COLORS.values())
for k in range(min(len(jets_m[i_show]), 5)):
    c = constituents[i_show, k]
    ax.scatter(ak.to_numpy(c.eta), ak.to_numpy(c.phi), s=4 + 3 * ak.to_numpy(c.pt),
               color=jet_colors[k], edgecolor="white", linewidth=0.5,
               label=f"jet {k} ({jets_m[i_show, k].pt:.0f} GeV, {len(c)} cands)")
    ax.add_patch(plt.Circle((jets_m[i_show, k].eta, jets_m[i_show, k].phi), 0.4,
                            fill=False, color="black"))
ax.set_xlim(-5, 5); ax.set_ylim(-np.pi, np.pi); ax.set_aspect("equal")
ax.set_xlabel(r"$\eta$"); ax.set_ylabel(r"$\phi$")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
ax.set_title(f"Event index {i_show}: candidates assigned to the 5 leading jets")
plt.show()

## 10. Checking the association, and studying jet constituents

How good is $\Delta R$ matching? NanoAOD gives us two independent cross-checks for every jet:

1. **`Jet_nConstituents`**: the number of PF candidates CMSSW actually clustered into the jet.
2. **The raw jet $p_T$**: the stored `Jet_pt` is *calibrated*. The uncalibrated value, which should equal the vector sum of the constituents, is
$$p_T^{\text{raw}} = p_T \times (1 - \texttt{rawFactor}).$$

We'll compare both against our matched constituents for **good** jets.

In [ ]:
# Four-vector sum over each jet's constituents (sum along the innermost axis)
px, py, pz, E = to_cartesian(constituents.pt, constituents.eta, constituents.phi, constituents.mass)
sum_px, sum_py = ak.sum(px, axis=-1), ak.sum(py, axis=-1)
sum_pz, sum_E  = ak.sum(pz, axis=-1), ak.sum(E, axis=-1)

jets_m = ak.with_field(jets_m, ak.num(constituents, axis=2), "nMatched")
jets_m = ak.with_field(jets_m, np.sqrt(sum_px**2 + sum_py**2), "ptFromCands")
jets_m = ak.with_field(jets_m, invariant_mass(sum_px, sum_py, sum_pz, sum_E), "massFromCands")
jets_m = ak.with_field(jets_m, jets_m.pt * (1 - jets_m.rawFactor), "ptRaw")

good_m = jets_m[(jets_m.pt > 30) & (abs(jets_m.eta) < 2.4) & ((jets_m.jetId & 2) > 0)]
print(good_m[:2, :3][["pt", "ptRaw", "ptFromCands", "nConstituents", "nMatched"]].tolist())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

n_true, n_match = to_np(good_m.nConstituents), to_np(good_m.nMatched)
top = np.percentile(n_true, 99.5)
h = axes[0].hist2d(n_true, n_match, bins=[np.arange(0, top + 2), np.arange(0, top * 1.5 + 2)],
                   cmap="Blues", cmin=1)
axes[0].plot([0, top], [0, top], color="#eb6834", linewidth=2, label="perfect agreement")
axes[0].set_xlabel("Jet_nConstituents (from CMSSW)")
axes[0].set_ylabel(r"# candidates matched with $\Delta R < 0.4$")
axes[0].legend(frameon=False, loc="upper left")
plt.colorbar(h[3], ax=axes[0], label="Jets")

ratio = to_np(good_m.ptFromCands / good_m.ptRaw)
axes[1].hist(ratio, bins=np.linspace(0.5, 1.5, 81), histtype="step", linewidth=2)
axes[1].axvline(1, color="#eb6834", linewidth=2)
axes[1].set_xlabel(r"$p_T$(matched candidates) / raw jet $p_T$")
axes[1].set_ylabel("Jets")
plt.tight_layout(); plt.show()

print(f"median pT ratio: {np.median(ratio):.3f};  "
      f"median (matched - true) count: {np.median(n_match - n_true):.0f}")

### Interpreting this
If $\Delta R$ matching were perfect, everything would sit on the orange line (left) and the ratio would be a sharp spike at 1 (right). It won't be perfect, and *understanding why* is the real physics lesson here:

* **CHS (charged hadron subtraction).** `Jet` is built *after* removing charged candidates from pileup vertices, but `PFCands` contains *all* candidates. So $\Delta R$ matching picks up extra charged pileup candidates, and the matched count comes out **above** `nConstituents`. (`Jet_chFPV0EF` stores the energy fraction CHS removed.)
* **Pileup and the underlying event** also contribute soft *neutral* candidates inside the cone that CMSSW did cluster, so those are counted correctly.
* **Jet shape.** Anti-$k_T$ jets are not perfect circles, especially when two jets are close together, so candidates near the edge can be gained or lost.
* **Precision.** NanoAOD stores many floats at reduced precision to save space, so small differences in $p_T$ are expected.

### A CHS-like cleanup (an approximation, for you to explore)
CMSSW removes a charged candidate from CHS jets when it isn't associated with the primary vertex. We don't have that exact flag in this file, but `PFCands_dz` (the track's longitudinal distance from the primary vertex, in cm) is a reasonable proxy. Let's drop charged candidates with large $|dz|$ and see whether the agreement improves.

In [ ]:
DZ_CUT = 0.2   # cm. A starting point, not an official CMS value. Experiment with it!

is_charged = pf_m.charge != 0
keep = (~is_charged) | (abs(pf_m.dz) < DZ_CUT)
constituents_chs = group_by_jet(jets_m, pf_m[keep])

px, py, _, _ = to_cartesian(constituents_chs.pt, constituents_chs.eta,
                            constituents_chs.phi, constituents_chs.mass)
jets_m = ak.with_field(jets_m, ak.num(constituents_chs, axis=2), "nMatchedCHS")
jets_m = ak.with_field(jets_m, np.sqrt(ak.sum(px, -1)**2 + ak.sum(py, -1)**2), "ptFromCandsCHS")
good_m = jets_m[(jets_m.pt > 30) & (abs(jets_m.eta) < 2.4) & ((jets_m.jetId & 2) > 0)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
bins_n = np.arange(-30.5, 30.5, 1)
axes[0].hist(to_np(good_m.nMatched - good_m.nConstituents), bins=bins_n, histtype="step",
             linewidth=2, label="all candidates", color="#2a78d6")
axes[0].hist(to_np(good_m.nMatchedCHS - good_m.nConstituents), bins=bins_n, histtype="step",
             linewidth=2, label=f"charged need |dz| < {DZ_CUT} cm", color="#eb6834")
axes[0].set_xlabel("matched count − Jet_nConstituents"); axes[0].set_ylabel("Jets")
axes[0].legend(frameon=False)

bins_r = np.linspace(0.5, 1.5, 81)
axes[1].hist(to_np(good_m.ptFromCands / good_m.ptRaw), bins=bins_r, histtype="step",
             linewidth=2, label="all candidates", color="#2a78d6")
axes[1].hist(to_np(good_m.ptFromCandsCHS / good_m.ptRaw), bins=bins_r, histtype="step",
             linewidth=2, label=f"charged need |dz| < {DZ_CUT} cm", color="#eb6834")
axes[1].set_xlabel(r"$p_T$(matched) / raw jet $p_T$"); axes[1].set_ylabel("Jets")
axes[1].legend(frameon=False)
plt.tight_layout(); plt.show()

**✏️ Try it:** Vary `DZ_CUT` (0.05, 0.1, 0.5, 1.0 cm). Which value gives the best agreement? Is there a cut that *overshoots*? Also look at `PFCands_pvAssocQuality` and `PFCands_puppiWeight` (use `tree.show` to read their descriptions). Could either one do the job better?

### Studying the constituents
Now that we have jets as lists of particles, we can ask questions that standard NanoAOD can't answer. Some examples:

In [ ]:
# Use the CHS-like constituents of good jets
good_sel = (jets_m.pt > 30) & (abs(jets_m.eta) < 2.4) & ((jets_m.jetId & 2) > 0)
gj   = jets_m[good_sel]
gcon = constituents_chs[good_sel]          # [event][good jet][candidate]

# (1) charged vs neutral multiplicity
n_charged = ak.sum(gcon.charge != 0, axis=-1)
n_neutral = ak.sum(gcon.charge == 0, axis=-1)

# (2) how far each constituent is from the jet axis
#     broadcasting: gj.eta has shape [event][jet], gcon.eta has [event][jet][cand]
dr_axis = delta_r(gcon.eta, gcon.phi, gj.eta, gj.phi)

# (3) pT fraction carried by the hardest constituent
lead_frac = ak.max(gcon.pt, axis=-1) / ak.sum(gcon.pt, axis=-1)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
b = np.arange(-0.5, 60.5, 1)
axes[0].hist(to_np(n_charged), bins=b, histtype="step", linewidth=2, color="#2a78d6", label="charged")
axes[0].hist(to_np(n_neutral), bins=b, histtype="step", linewidth=2, color="#eb6834", label="neutral")
axes[0].set_xlabel("Constituents per jet"); axes[0].set_ylabel("Jets"); axes[0].legend(frameon=False)

hist1d(dr_axis, np.linspace(0, 0.4, 41), r"$\Delta R$(constituent, jet axis)", ax=axes[1],
       weights=to_np(gcon.pt))
axes[1].set_ylabel(r"$\sum p_T$ [GeV]")
axes[1].set_title(r"$p_T$-weighted: energy is concentrated near the axis", fontsize=11)

hist1d(lead_frac, np.linspace(0, 1, 41), r"leading constituent $p_T$ / jet $p_T$", ax=axes[2],
       ylabel="Jets")
plt.tight_layout(); plt.show()

The middle plot is a simple **jet shape**: most of a jet's energy lies close to its axis. Quark-initiated jets are typically narrower, with fewer constituents, than gluon-initiated jets. That's the idea behind quark/gluon taggers such as `Jet_qgl`.

**✏️ Try it:** Compare `massFromCands` with `Jet_mass`. Then make the constituent-multiplicity plot separately for jets with $p_T$ in 30–100 GeV and above 300 GeV. How does the multiplicity change with $p_T$?

## 11. Running over the whole file in chunks

The full file is hundreds of MB, and loading every PF candidate at once can use many GB of memory. The standard pattern is to **iterate in chunks** and **accumulate histograms**. `uproot.iterate` reads the file a piece at a time; you do the same analysis on each piece and add up the histogram counts.

`np.histogram` returns counts, which you can add across chunks, and `plt.stairs` draws them.

In [ ]:
bins_pt = np.linspace(0, 1000, 101)
counts_lead_pt = np.zeros(len(bins_pt) - 1)
counts_nmatched = np.zeros(80)
n_total = 0

t0 = time.time()
for chunk in uproot.iterate(
        f"{FILENAME}:Events",
        filter_name=["Jet_pt", "Jet_eta", "Jet_phi", "Jet_jetId", "Jet_nConstituents",
                     "PFCands_pt", "PFCands_eta", "PFCands_phi", "PFCands_charge", "PFCands_dz"],
        step_size=500):     # events per chunk. The jet x candidate pairing is memory-hungry;
                            # lower this if you run out of memory, raise it if you have plenty

    j  = get_collection(chunk, "Jet")
    pf = get_collection(chunk, "PFCands")
    n_total += len(chunk)

    # leading good jet pT
    good = j[(j.pt > 30) & (abs(j.eta) < 2.4) & ((j.jetId & 2) > 0)]
    lead = ak.firsts(good.pt)
    counts_lead_pt += np.histogram(ak.to_numpy(lead[~ak.is_none(lead)]), bins=bins_pt)[0]

    # Delta-R association (CHS-like) for every chunk
    pf = pf[(pf.charge == 0) | (abs(pf.dz) < DZ_CUT)]
    idx, _ = match_pf_to_jets(pf, j)
    con = group_by_jet(j, ak.with_field(pf, idx, "jetIdx"))
    good_mask = (j.pt > 30) & (abs(j.eta) < 2.4) & ((j.jetId & 2) > 0)
    counts_nmatched += np.histogram(to_np(ak.num(con, axis=2)[good_mask]),
                                    bins=np.arange(81) - 0.5)[0]

    print(f"  {n_total} events processed ({time.time() - t0:.0f} s)", end="\r")

print(f"\nDone: {n_total} events in {time.time() - t0:.0f} s")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].stairs(counts_lead_pt, bins_pt, linewidth=2)
axes[0].set_yscale("log"); axes[0].set_xlabel(r"leading good jet $p_T$ [GeV]"); axes[0].set_ylabel("Events")
axes[1].stairs(counts_nmatched, np.arange(81) - 0.5, linewidth=2)
axes[1].set_xlabel("matched constituents per good jet"); axes[1].set_ylabel("Jets")
plt.tight_layout(); plt.show()

> If this is slow or runs out of memory, lower `step_size` or add `entry_stop=50000` to `uproot.iterate`. For multi-file, multi-core processing, people eventually move to tools like [`coffea`](https://coffea-hep.readthedocs.io/) or [Dask](https://docs.dask.org/), but the chunked loop above is the core idea behind all of them.

### Saving results
Awkward arrays can be saved to disk and reloaded quickly, for example as [Parquet](https://awkward-array.org/doc/main/reference/generated/ak.to_parquet.html):
```python
ak.to_parquet(good_m, "good_jets_sample.parquet")
again = ak.from_parquet("good_jets_sample.parquet")
```

## 12. More exercises

1. **Jet energy fractions.** NanoAOD stores the charged-hadron energy fraction `Jet_chHEF` for every jet. Compute the same quantity from your matched constituents (energy of `|pdgId| == 211` constituents ÷ total constituent energy) and compare. Do the same for photons vs `Jet_neEmEF`.
2. **Trigger.** Which `HLT_PFJet*` triggers fired in these events? Plot the leading-jet $p_T$ for events passing `HLT_PFJet450` and see where the "turn-on" is.
3. **Pileup.** Plot the average number of PF candidates per event vs `PV_npvs` (the number of reconstructed primary vertices). What does the trend tell you?
4. **Matching radius.** Repeat the association with $R = 0.3$ and $R = 0.5$. How do the comparisons in Section 10 change? Why is 0.4 the natural choice?
5. **Fat jets.** Repeat the association for `FatJet` with $R = 0.8$. (Candidates can legitimately belong to both an AK4 jet *and* an AK8 jet, so treat the two collections separately.)
6. **Looking ahead: reclustering.** The next step is to run the anti-$k_T$ algorithm on the PF candidates yourself with [FastJet](http://fastjet.fr/) via the [`fastjet` Python package](https://fastjet.readthedocs.io/) (`pip install fastjet`), which takes awkward arrays directly. Once you recluster, you get the *exact* constituent list for your own jets, and you can compare your jets with the CMS `Jet` collection using everything you built here.

## 13. References and further reading

**The data**
* [CMS Open Data: Getting started with NanoAOD](https://opendata.cern.ch/docs/cms-getting-started-nanoaod)
* [PFNano producer tool for CMS 2016 Open Data (record 12504)](https://opendata.cern.ch/record/12504) and its [GitHub repository](https://github.com/cms-opendata-analyses/PFNanoProducerTool)
* [Branch-by-branch documentation of a 2016 PFNano data file](https://htmlpreview.github.io/?https://github.com/cms-opendata-analyses/PFNanoProducerTool/blob/master/nanoAOD_2016data.html)
* [CMS NanoAOD documentation for standard branches](https://cms-nanoaod-integration.web.cern.ch/autoDoc/)
* [CMS Open Data Guide](https://cms-opendata-guide.web.cern.ch/)

**The tools**
* [uproot documentation](https://uproot.readthedocs.io/) — especially "Getting started"
* [Awkward Array: Getting started](https://awkward-array.org/doc/main/getting-started/index.html) and the [user guide](https://awkward-array.org/doc/main/user-guide/index.html)
* [CMS Open Data Workshop 2022: awkward arrays lesson](https://cms-opendata-workshop.github.io/workshop2022-lesson-cpp-root-python/08-awkward/)
* [HEP Software Foundation tutorial on uproot + awkward](https://hsf-training.github.io/hsf-training-uproot-webpage/)
* [`vector`](https://vector.readthedocs.io/) (four-vectors on awkward arrays), [`hist`](https://hist.readthedocs.io/), and [`mplhep`](https://mplhep.readthedocs.io/)
* For reference, how `coffea` handles the same data: [`PFNanoAODSchema`](https://coffea-hep.readthedocs.io/en/v0.7.23/api/coffea.nanoevents.PFNanoAODSchema.html) and [`AssociatedPFCand`](https://scikit-hep.org/coffea/api/coffea.nanoevents.methods.nanoaod.AssociatedPFCand.html)

**The physics**
* CMS Collaboration, "Particle-flow reconstruction and global event description with the CMS detector," [JINST 12 (2017) P10003](https://doi.org/10.1088/1748-0221/12/10/P10003), [arXiv:1706.04965](https://arxiv.org/abs/1706.04965)
* M. Cacciari, G. Salam, G. Soyez, "The anti-$k_t$ jet clustering algorithm," [JHEP 04 (2008) 063](https://doi.org/10.1088/1126-6708/2008/04/063), [arXiv:0802.1189](https://arxiv.org/abs/0802.1189)
* G. Salam, "Towards jetography," [Eur. Phys. J. C 67 (2010) 637](https://doi.org/10.1140/epjc/s10052-010-1314-6), [arXiv:0906.1833](https://arxiv.org/abs/0906.1833), a readable introduction to jets
* CMS Collaboration, "Pileup mitigation at CMS in 13 TeV data," [JINST 15 (2020) P09018](https://doi.org/10.1088/1748-0221/15/09/P09018), [arXiv:2003.00503](https://arxiv.org/abs/2003.00503) (CHS and PUPPI)
* [PDG Monte Carlo particle numbering scheme](https://pdg.lbl.gov/2024/reviews/rpp2024-rev-monte-carlo-numbering.pdf) (what the `pdgId` numbers mean)